In [6]:
!pip install duckdb

In [8]:
import duckdb

# 1. KHAI BÁO THÔNG TIN R2
ACCESS_KEY = "352e66be1b3cb7fad1866c153222faf2"
SECRET_KEY = "6b97ebc8310262f21fcb5e1d315ce7b3224f73a39f7813ebb38c21fa4ab30ea2"
ACCOUNT_ID = "02422b2aa3d6c7acdf824dfd9259afb2"

R2_ENDPOINT = f"{ACCOUNT_ID}.r2.cloudflarestorage.com"
BUCKET_NAME = "amazon-reviews-test"
CATEGORY = "Tools_and_Home_Improvement"

source_path = f"s3://{BUCKET_NAME}/parquet-data/meta/{CATEGORY}/*.parquet"
dest_path = f"s3://{BUCKET_NAME}/data-preprocess/meta/{CATEGORY}/processed_meta.parquet"

# 2. KHỞI TẠO KẾT NỐI DUCKDB VÀ CẤU HÌNH S3
print("⏳ Đang thiết lập DuckDB và kết nối R2...")
con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("SET s3_region='auto';")
con.execute(f"SET s3_endpoint='{R2_ENDPOINT}';")
con.execute(f"SET s3_access_key_id='{ACCESS_KEY}';")
con.execute(f"SET s3_secret_access_key='{SECRET_KEY}';")

# 3. TIỀN XỬ LÝ VÀ GHI DỮ LIỆU
print("⚙️ Đang thực thi tiền xử lý trực tiếp trên R2 (Out-of-Core)...")

# Hàm Python hỗ trợ sinh chuỗi SQL an toàn tuyệt đối
# Xóa 4 ký tự rác: [, ], dấu nháy đơn ('), dấu nháy kép (")
def clean_sql(col_name):
    return f"REPLACE(REPLACE(REPLACE(REPLACE(COALESCE({col_name}, ''), '[', ''), ']', ''), '''', ''), '\"', '')"

query = f"""
COPY (
    SELECT 
        parent_asin,
        TRY_CAST(average_rating AS FLOAT) AS average_rating,
        TRY_CAST(rating_number AS INTEGER) AS rating_number,
        TRY_CAST(price AS FLOAT) AS price,
        store,
        
        {clean_sql('title')} AS clean_title,
        {clean_sql('categories')} AS clean_categories,
        SUBSTRING({clean_sql('features')}, 1, 500) AS clean_features,
        SUBSTRING({clean_sql('description')}, 1, 1000) AS clean_description,
        
        -- Nối tất cả lại thành 1 cột văn bản duy nhất
        CONCAT_WS(' ', 
            {clean_sql('title')},
            {clean_sql('categories')},
            SUBSTRING({clean_sql('features')}, 1, 500),
            SUBSTRING({clean_sql('description')}, 1, 1000)
        ) AS combined_text
    FROM read_parquet('{source_path}')
    WHERE parent_asin IS NOT NULL
) TO '{dest_path}' (FORMAT 'parquet');
"""

try:
    con.execute(query)
    print("✅ QUÁ TRÌNH TIỀN XỬ LÝ BẰNG DUCKDB HOÀN TẤT!")
except Exception as e:
    print(f"❌ Có lỗi xảy ra: {e}")

⏳ Đang thiết lập DuckDB và kết nối R2...
⚙️ Đang thực thi tiền xử lý trực tiếp trên R2 (Out-of-Core)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ QUÁ TRÌNH TIỀN XỬ LÝ BẰNG DUCKDB HOÀN TẤT!


In [9]:
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.sql.functions import col

# 1. CẤU HÌNH HỆ THỐNG (ĐÃ FIX TẤT CẢ LỖI TRƯỚC ĐÓ)
ACCESS_KEY = "352e66be1b3cb7fad1866c153222faf2"
SECRET_KEY = "6b97ebc8310262f21fcb5e1d315ce7b3224f73a39f7813ebb38c21fa4ab30ea2"
ACCOUNT_ID = "02422b2aa3d6c7acdf824dfd9259afb2"

R2_ENDPOINT = f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com"
BUCKET_NAME = "amazon-reviews-test"
CATEGORY = "Tools_and_Home_Improvement"

# Đường dẫn file meta duy nhất bạn vừa tạo bằng DuckDB
source_meta = f"s3a://{BUCKET_NAME}/data-preprocess/meta/{CATEGORY}/processed_meta.parquet"
# Đường dẫn lưu ma trận đặc trưng sau TF-IDF
dest_tfidf = f"s3a://{BUCKET_NAME}/data-preprocess/meta/{CATEGORY}/tfidf_vectors.parquet"

print("⏳ Đang khởi tạo Spark với cấu hình tối ưu 16GB RAM...")
spark = SparkSession.builder \
    .appName("TF-IDF_Feature_Extraction") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.driver.memory", "16g") \
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.endpoint", R2_ENDPOINT) \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.request.timeout", "30000") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

# 2. ĐỌC DỮ LIỆU
print(f"📥 Đang nạp dữ liệu meta từ R2...")
df_meta = spark.read.parquet(source_meta)

# 3. PIPELINE TRÍCH XUẤT ĐẶC TRƯNG
print("⚙️ Đang thực hiện Pipeline TF-IDF (Tokenizer -> StopWords -> TF -> IDF)...")

# Bước A: Tách từ (Tokenizer)
tokenizer = Tokenizer(inputCol="combined_text", outputCol="words")
words_data = tokenizer.transform(df_meta)

# Bước B: Loại bỏ từ dừng (StopWordsRemover - xóa 'the', 'is', 'a'...)
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
filtered_data = remover.transform(words_data)

# Bước C: Tính Term Frequency (TF) bằng HashingTF (nhanh và tiết kiệm RAM)
# numFeatures=10000 là kích thước vector (độ dài DNA của sản phẩm)
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=10000)
featurized_data = hashingTF.transform(filtered_data)

# Bước D: Tính Inverse Document Frequency (IDF) để nhấn mạnh các từ khóa hiếm/đặc trưng
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurized_data)
rescaled_data = idfModel.transform(featurized_data)

# 4. CHỌN CỘT VÀ LƯU KẾT QUẢ
# Chúng ta chỉ giữ lại parent_asin và cột features (vector số) để tiết kiệm dung lượng
print("📤 Đang lưu ma trận TF-IDF Vectors lên R2...")
final_tfidf = rescaled_data.select("parent_asin", "features")
final_tfidf.write.mode("overwrite").parquet(dest_tfidf)

print(f"✅ HOÀN TẤT! Vector sản phẩm đã được lưu tại: {dest_tfidf}")

⏳ Đang khởi tạo Spark với cấu hình tối ưu 16GB RAM...
📥 Đang nạp dữ liệu meta từ R2...


⚙️ Đang thực hiện Pipeline TF-IDF (Tokenizer -> StopWords -> TF -> IDF)...


📤 Đang lưu ma trận TF-IDF Vectors lên R2...


✅ HOÀN TẤT! Vector sản phẩm đã được lưu tại: s3a://amazon-reviews-test/data-preprocess/meta/Tools_and_Home_Improvement/tfidf_vectors.parquet


In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.ml.linalg import Vectors, VectorUDT, SparseVector, DenseVector
from pyspark.ml.stat import Summarizer

# =====================================================================
# 1. GHI TRỰC TIẾP THÔNG TIN XÁC THỰC R2 (HARDCODE)
# =====================================================================
R2_ACCESS_KEY = "352e66be1b3cb7fad1866c153222faf2"
R2_SECRET_KEY = "6b97ebc8310262f21fcb5e1d315ce7b3224f73a39f7813ebb38c21fa4ab30ea2"
R2_ACCOUNT_ID = "02422b2aa3d6c7acdf824dfd9259afb2"

# =====================================================================
# 2. KHỞI TẠO SPARK SESSION (TỐI ƯU RAM KAGGLE + FIX LỖI S3A)
# =====================================================================
spark = SparkSession.builder \
    .appName("Step2_User_Profiling_Sample") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com") \
    .config("spark.hadoop.fs.s3a.access.key", R2_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", R2_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "200000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.paging.maximum", "5000") \
    .config("spark.hadoop.fs.s3a.attempts.maximum", "20") \
    .config("spark.hadoop.fs.s3a.retry.limit", "20") \
    .config("spark.hadoop.fs.s3a.max.total.tasks", "1000") \
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
    .config("spark.hadoop.fs.s3a.multipart.purge", "false") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    .getOrCreate()

# =====================================================================
# 3. ĐỊNH NGHĨA ĐƯỜNG DẪN R2
# =====================================================================
BUCKET_NAME = "amazon-reviews-test"
CATEGORY = "Tools_and_Home_Improvement"

review_path = f"s3a://{BUCKET_NAME}/data-preprocess/review/{CATEGORY}/"
tfidf_path = f"s3a://{BUCKET_NAME}/data-preprocess/meta/{CATEGORY}/tfidf_vectors.parquet"
# Đổi tên thư mục output để phân biệt đây là bản chạy 20%
output_path = f"s3a://{BUCKET_NAME}/data-preprocess/user_profiles_sample_20/{CATEGORY}/"

# =====================================================================
# 4. ĐỌC DỮ LIỆU & LẤY MẪU 20% (1/5) USER
# =====================================================================
print("Đang đọc dữ liệu từ R2...")
df_review_full = spark.read.parquet(review_path).select("user_id", "parent_asin", "rating")
df_tfidf = spark.read.parquet(tfidf_path).select("parent_asin", col("features").alias("tfidf_features"))

# A. Lấy danh sách toàn bộ User độc nhất
df_unique_users = df_review_full.select("user_id").distinct()

# B. Lấy mẫu ngẫu nhiên 20% số lượng User (đặt seed để kết quả ổn định)
df_sampled_users = df_unique_users.sample(withReplacement=False, fraction=0.2, seed=42)

# C. Lọc lại tập Review ban đầu, chỉ giữ lại đánh giá của 20% User này
df_review = df_review_full.join(df_sampled_users, on="user_id", how="inner")
print("Đã lấy mẫu xong 20% dữ liệu User để xử lý!")

# =====================================================================
# 5. UDF: NHÂN VECTOR VỚI RATING
# =====================================================================
def multiply_vector_by_rating(vec, rating):
    if vec is None or rating is None:
        return None
    if isinstance(vec, SparseVector):
        # Nhân các giá trị khác 0 với rating
        return SparseVector(vec.size, vec.indices, vec.values * rating)
    elif isinstance(vec, DenseVector):
        return DenseVector(vec.toArray() * rating)

# Đăng ký UDF
multiply_vec_udf = udf(multiply_vector_by_rating, VectorUDT())

# =====================================================================
# 6. XỬ LÝ DỮ LIỆU: JOIN -> NHÂN -> TÍNH TỔNG (TẠO PROFILE)
# =====================================================================
print("Đang tiến hành tính toán User Profile...")
# Bước A: Join Review và TF-IDF
df_joined = df_review.join(df_tfidf, on="parent_asin", how="inner")

# Bước B: Tính Product_Vector * User_Rating
df_weighted = df_joined.withColumn(
    "weighted_vector", 
    multiply_vec_udf(col("tfidf_features"), col("rating"))
)

# Bước C: Group by User và tính tổng các vector (Profile)
df_user_profiles = df_weighted.groupBy("user_id").agg(
    Summarizer.sum(col("weighted_vector")).alias("user_profile_vector")
)

# =====================================================================
# 7. GHI KẾT QUẢ RA CLOUDFLARE R2
# =====================================================================
print(f"Đang ghi dữ liệu User Profiles (Sample 20%) lên R2 tại:\n{output_path}")
df_user_profiles.write \
    .mode("overwrite") \
    .parquet(output_path)

print("Hoàn thành xuất sắc! Bạn có thể kiểm tra bucket R2 rồi nhé.")

# Dừng Spark Session để giải phóng tài nguyên RAM
spark.stop()

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-766b4ace-de1f-4a9f-93a0-b90bc2b3ea3b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (118ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws

Đang đọc dữ liệu từ R2...


26/03/31 20:36:56 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Đã lấy mẫu xong 20% dữ liệu User để xử lý!
Đang tiến hành tính toán User Profile...
Đang ghi dữ liệu User Profiles (Sample 20%) lên R2 tại:
s3a://amazon-reviews-test/data-preprocess/user_profiles_sample_20/Tools_and_Home_Improvement/


26/03/31 21:02:34 WARN Base64: JAXB is unavailable. Will fallback to SDK implementation which may be less performant.If you are using Java 9+, you will need to include javax.xml.bind:jaxb-api as a dependency.
26/03/31 21:03:22 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to data-preprocess/user_profiles_sample_20/Tools_and_Home_Improvement/_temporary/0/_temporary/attempt_20260331210234617386729054494759_0018_m_000001_141/part-00001-f57ada0e-8750-47c1-a912-c6b196438b35-c000.snappy.parquet. This is unsupported
ERROR:root:Exception while sending command.                                     
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: reentrant call inside <_io.BufferedReader name=59>

During handling of the above exception, another exception occ

Py4JError: An error occurred while calling o141.parquet

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, collect_list, struct
from pyspark.sql.window import Window
from pyspark.ml.feature import Normalizer, BucketedRandomProjectionLSH

# =====================================================================
# 1. THÔNG TIN XÁC THỰC R2 (Nhớ điền Key của bạn vào đây)
# =====================================================================
R2_ACCESS_KEY = "352e66be1b3cb7fad1866c153222faf2"
R2_SECRET_KEY = "6b97ebc8310262f21fcb5e1d315ce7b3224f73a39f7813ebb38c21fa4ab30ea2"
R2_ACCOUNT_ID = "02422b2aa3d6c7acdf824dfd9259afb2"
# =====================================================================
# 2. KHỞI TẠO SPARK (Tối ưu RAM, Disk và Fix TOÀN BỘ lỗi String S3A)
# =====================================================================
spark = SparkSession.builder \
    .appName("Step3_Recommendation_LSH") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.local.dir", "/kaggle/working/spark-temp") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com") \
    .config("spark.hadoop.fs.s3a.access.key", R2_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", R2_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "200000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.paging.maximum", "5000") \
    .config("spark.hadoop.fs.s3a.attempts.maximum", "20") \
    .config("spark.hadoop.fs.s3a.retry.limit", "20") \
    .config("spark.hadoop.fs.s3a.max.total.tasks", "1000") \
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
    .config("spark.hadoop.fs.s3a.multipart.purge", "false") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    .getOrCreate()

# =====================================================================
# 3. ĐỊNH NGHĨA ĐƯỜNG DẪN R2
# =====================================================================
BUCKET_NAME = "amazon-reviews-test"
CATEGORY = "Tools_and_Home_Improvement"

# Đầu vào
user_profile_path = f"s3a://{BUCKET_NAME}/data-preprocess/user_profiles_sample_20/{CATEGORY}/"
tfidf_path = f"s3a://{BUCKET_NAME}/data-preprocess/meta/{CATEGORY}/tfidf_vectors.parquet"

# Đầu ra
output_recommendations = f"s3a://{BUCKET_NAME}/data-preprocess/top10_recommendations/{CATEGORY}/"

# =====================================================================
# 4. ĐỌC DỮ LIỆU
# =====================================================================
print("Đang đọc User Profiles và Item Vectors...")
df_users = spark.read.parquet(user_profile_path)
df_items = spark.read.parquet(tfidf_path).select("parent_asin", col("features").alias("tfidf_features"))

# =====================================================================
# 5. CHUẨN HÓA VECTOR (Đồng bộ tên cột thành norm_features)
# =====================================================================
normalizer_user = Normalizer(p=2.0, inputCol="user_profile_vector", outputCol="norm_features")
df_users_norm = normalizer_user.transform(df_users)

normalizer_item = Normalizer(p=2.0, inputCol="tfidf_features", outputCol="norm_features")
df_items_norm = normalizer_item.transform(df_items)

# =====================================================================
# 6. THUẬT TOÁN LSH (Siết chặt cấu hình chống tràn RAM)
# =====================================================================
print("Đang huấn luyện mô hình băm LSH...")
brp = BucketedRandomProjectionLSH(
    inputCol="norm_features", 
    outputCol="hashes", 
    bucketLength=1.0,  # Xô hẹp hơn để lọc kỹ đồ vớ vẩn ngay từ đầu
    numHashTables=1    # Giảm tải cho Kaggle
)
lsh_model = brp.fit(df_items_norm)

# Caching dữ liệu lên RAM để hàm Join chạy siêu tốc
df_users_norm.cache()
df_items_norm.cache()

# =====================================================================
# 7. TÌM KIẾM CẶP TƯƠNG ĐỒNG (Similarity Join)
# =====================================================================
print("Đang ghép cặp User và Item (Similarity Join)...")
# Siết ngưỡng threshold=1.0 để dọn sạch 99% lượng file rác
df_joined = lsh_model.approxSimilarityJoin(
    df_users_norm, df_items_norm, threshold=1.0, distCol="EuclideanDistance"
)

# Rút gọn bảng
df_scored = df_joined.select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.parent_asin").alias("recommended_item"),
    col("EuclideanDistance") 
)

# =====================================================================
# 8. LỌC LẤY TOP 10 SẢN PHẨM CHO MỖI USER
# =====================================================================
print("Đang xếp hạng và lấy Top 10...")
windowSpec = Window.partitionBy("user_id").orderBy("EuclideanDistance")

df_top_k = df_scored.withColumn("rank", row_number().over(windowSpec)) \
    .filter(col("rank") <= 10)

# Gom vào 1 mảng JSON cho đẹp file đầu ra
df_final = df_top_k.groupBy("user_id").agg(
    collect_list(
        struct(
            col("recommended_item").alias("item"), 
            col("EuclideanDistance").alias("distance")
        )
    ).alias("top_10_recommendations")
)

# =====================================================================
# 9. GHI KẾT QUẢ RA R2
# =====================================================================
print(f"Đang ghi danh sách Top 10 Gợi ý lên R2 tại:\n{output_recommendations}")
df_final.write.mode("overwrite").parquet(output_recommendations)

print("XUẤT SẮC! Đồ án của bạn đã hoàn thiện luồng Content-Based Recommendation!")

# Dọn dẹp RAM
df_users_norm.unpersist()
df_items_norm.unpersist()
spark.stop()

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b27de6b8-894c-4dfa-99d0-46939a510f06;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (127ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-1.12.262.jar ...
	[SUCCESSFUL ] com.amazonaws#aws

Đang đọc User Profiles và Item Vectors...


26/04/01 00:01:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
